# State of Data Brasil — Análise de Gênero (2021–2024)

> **Objetivo:** Investigar desigualdades de gênero no mercado de dados brasileiro.  
> **Recorte:** Gênero × Remuneração, Faixa Salarial e Senioridade.  
> **Split ML:** Treino = 2021–2023 (com features de IA de 2023) · Teste = 2024 (com features de IA de 2024)


---
## Seção 0 — Imports e Carregamento da base de dados

In [26]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import os

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns

from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu
import statsmodels.api as sm

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from xgboost import XGBClassifier
import shap

os.makedirs('figures', exist_ok=True)
os.makedirs('data', exist_ok=True)

SEED  = 42
np.random.seed(SEED)
ALPHA = 0.05
print('OK')

OK


In [27]:
dfs_raw = {
    '2021': pd.read_csv('data/state_of_data_2021.csv', low_memory=False),
    '2022': pd.read_csv('data/state_of_data_2022.csv', low_memory=False),
    '2023': pd.read_csv('data/state_of_data_2023.csv', low_memory=False),
    '2024': pd.read_csv('data/state_of_data_2024.csv', low_memory=False),
}
for ano, df_raw in dfs_raw.items():
    print(f'{ano}: {df_raw.shape[0]:,} linhas × {df_raw.shape[1]} colunas')

2021: 2,645 linhas × 356 colunas
2022: 4,271 linhas × 353 colunas
2023: 5,293 linhas × 399 colunas
2024: 5,217 linhas × 403 colunas


### 0.1 Mapeamento de colunas por ano

In [28]:
COLUMN_MAPPING = {
    '2021': {
        'idade':            "('P1_a ', 'Idade')",
        'faixa_idade':      "('P1_a_a ', 'Faixa idade')",
        'genero':           "('P1_b ', 'Genero')",
        'estado':           "('P1_e ', 'Estado onde mora')",
        'uf':               "('P1_e_a ', 'uf onde mora')",
        'regiao':           "('P1_e_b ', 'Regiao onde mora')",
        'regiao_origem':    "('P1_g_b ', 'Regiao de origem')",
        'nivel_ensino':     "('P1_h ', 'Nivel de Ensino')",
        'area_formacao':    "('P1_i ', 'Área de Formação')",
        'situacao':         "('P2_a ', 'Qual sua situação atual de trabalho?')",
        'setor':            "('P2_b ', 'Setor')",
        'cargo':            "('P2_f ', 'Cargo Atual')",
        'gestor':           "('P2_d ', 'Gestor?')",
        'nivel':            "('P2_g ', 'Nivel')",
        'faixa_salarial':   "('P2_h ', 'Faixa salarial')",
        'tempo_area_dados': "('P2_i ', 'Quanto tempo de experiência na área de dados você tem?')",
        'tempo_area_ti':    "('P2_j ', 'Quanto tempo de experiência na área de TI/Engenharia de Software você teve antes de começar a trabalhar na área de dados?')",
        'modalidade':       "('P2_q ', 'Atualmente qual a sua forma de trabalho?')",
    },
    '2022': {
        'idade':            "('P1_a ', 'Idade')",
        'faixa_idade':      "('P1_a_1 ', 'Faixa idade')",
        'genero':           "('P1_b ', 'Genero')",
        'estado':           "('P1_i ', 'Estado onde mora')",
        'uf':               "('P1_i_1 ', 'uf onde mora')",
        'regiao':           "('P1_i_2 ', 'Regiao onde mora')",
        'regiao_origem':    "('P1_k ', 'Regiao de origem')",
        'nivel_ensino':     "('P1_l ', 'Nivel de Ensino')",
        'area_formacao':    "('P1_m ', 'Área de Formação')",
        'situacao':         "('P2_a ', 'Qual sua situação atual de trabalho?')",
        'setor':            "('P2_b ', 'Setor')",
        'cargo':            "('P2_f ', 'Cargo Atual')",
        'gestor':           "('P2_d ', 'Gestor?')",
        'nivel':            "('P2_g ', 'Nivel')",
        'faixa_salarial':   "('P2_h ', 'Faixa salarial')",
        'tempo_area_dados': "('P2_i ', 'Quanto tempo de experiência na área de dados você tem?')",
        'tempo_area_ti':    "('P2_j ', 'Quanto tempo de experiência na área de TI/Engenharia de Software você teve antes de começar a trabalhar na área de dados?')",
        'modalidade':       "('P2_p ', 'Atualmente qual a sua forma de trabalho?')",
    },
    '2023': {
        'idade':            "('P1_a ', 'Idade')",
        'faixa_idade':      "('P1_a_1 ', 'Faixa idade')",
        'genero':           "('P1_b ', 'Genero')",
        'estado':           "('P1_i ', 'Estado onde mora')",
        'uf':               "('P1_i_1 ', 'uf onde mora')",
        'regiao':           "('P1_i_2 ', 'Regiao onde mora')",
        'regiao_origem':    "('P1_k ', 'Regiao de origem')",
        'nivel_ensino':     "('P1_l ', 'Nivel de Ensino')",
        'area_formacao':    "('P1_m ', 'Área de Formação')",
        'situacao':         "('P2_a ', 'Qual sua situação atual de trabalho?')",
        'setor':            "('P2_b ', 'Setor')",
        'cargo':            "('P2_f ', 'Cargo Atual')",
        'gestor':           "('P2_d ', 'Gestor?')",
        'nivel':            "('P2_g ', 'Nivel')",
        'faixa_salarial':   "('P2_h ', 'Faixa salarial')",
        'tempo_area_dados': "('P2_i ', 'Quanto tempo de experiência na área de dados você tem?')",
        'tempo_area_ti':    "('P2_j ', 'Quanto tempo de experiência na área de TI/Engenharia de Software você teve antes de começar a trabalhar na área de dados?')",
        'modalidade':       "('P2_r ', 'Atualmente qual a sua forma de trabalho?')",
    },
    '2024': {
        'idade':            '1.a_idade',
        'faixa_idade':      '1.a.1_faixa_idade',
        'genero':           '1.b_genero',
        'estado':           '1.i_estado_onde_mora',
        'uf':               '1.i.1_uf_onde_mora',
        'regiao':           '1.i.2_regiao_onde_mora',
        'regiao_origem':    '1.k.2_regiao_de_origem',
        'nivel_ensino':     '1.l_nivel_de_ensino',
        'area_formacao':    '1.m_área_de_formação',
        'situacao':         '2.a_situação_de_trabalho',
        'setor':            '2.b_setor',
        'cargo':            '2.f_cargo_atual',
        'gestor':           '2.d_atua_como_gestor',
        'nivel':            '2.g_nivel',
        'faixa_salarial':   '2.h_faixa_salarial',
        'tempo_area_dados': '2.i_tempo_de_experiencia_em_dados',
        'tempo_area_ti':    '2.j_tempo_de_experiencia_em_ti',
        'modalidade':       '2.r_modelo_de_trabalho_atual',
    }
}

IA_COLS = {
    '2023': {
        'ia_prioridade_empresa':      "('P3_e ', 'AI Generativa é uma prioridade em sua empresa?')",
        'ia_tipos_uso_gestao':        "('P3_f ', 'Tipos de uso de AI Generativa e LLMs na empresa')",
        'ia_motivos_nao_usar':        "('P3_g ', 'Motivos que levam a empresa a não usar AI Genrativa e LLMs')",
        'ia_tipo_uso_pessoal':        "('P4_l ', 'Qual o tipo de uso de AI Generativa e LLMs na empresa')",
        'ia_usa_chatgpt_no_trabalho': "('P4_m ', 'Utiliza ChatGPT ou LLMs no trabalho?')",
    },
    '2024': {
        'ia_prioridade_empresa':      '3.e_ai_generativa_e_llm_é_uma_prioridade?',
        'ia_tipos_uso_gestao':        '3.f_tipo_de_uso_de_ai_generativa_e_llm_na_empresa',
        'ia_motivos_nao_usar':        '3.g_motivos_para_não_usar_ai_generativa_e_llm',
        'ia_tipo_uso_pessoal':        '4.l_tipo_de_uso_de_ai_generativa_e_llm_na_empresa',
        'ia_usa_chatgpt_no_trabalho': '4.m_usa_chatgpt_ou_copilot_no_trabalho?',
    },
}

IA_FEATS_ALL = [
    'ia_prioridade_empresa',
    'ia_tipos_uso_gestao',
    'ia_motivos_nao_usar',
    'ia_tipo_uso_pessoal',
    'ia_usa_chatgpt_no_trabalho'
]

### 0.2 Mapeamentos ordinais e de categorias

In [29]:
FAIXA_SALARIAL_MAP = {
    'de R$ 1.001/mês a R$ 2.000/mês':   'R$1k-2k',
    'de R$ 2.001/mês a R$ 3.000/mês':   'R$2k-3k',
    'de R$ 2.001/mês a R$ 3000/mês':    'R$2k-3k',
    'de R$ 3.001/mês a R$ 4.000/mês':   'R$3k-4k',
    'de R$ 4.001/mês a R$ 6.000/mês':   'R$4k-6k',
    'de R$ 6.001/mês a R$ 8.000/mês':   'R$6k-8k',
    'de R$ 8.001/mês a R$ 12.000/mês':  'R$8k-12k',
    'de R$ 12.001/mês a R$ 16.000/mês': 'R$12k-16k',
    'de R$ 16.001/mês a R$ 20.000/mês': 'R$16k-20k',
    'de R$ 20.001/mês a R$ 25.000/mês': 'R$20k-25k',
    'de R$ 25.001/mês a R$ 30.000/mês': 'R$25k-30k',
    'de R$ 30.001/mês a R$ 40.000/mês': 'R$30k-40k',
    'Acima de R$ 40.001/mês':           'R$40k+',
}
FAIXA_SALARIAL_ORDEM = [
    'R$1k-2k', 'R$2k-3k', 'R$3k-4k', 'R$4k-6k', 'R$6k-8k',
    'R$8k-12k', 'R$12k-16k', 'R$16k-20k', 'R$20k-25k',
    'R$25k-30k', 'R$30k-40k', 'R$40k+',
]
NIVEL_ENSINO_ORDEM = [
    'Não tenho graduação formal', 'Estudante de Graduação',
    'Graduação/Bacharelado', 'Especialização Lato Sensu',
    'Mestrado', 'Doutorado ou Phd',
]
NIVEL_SENIORIDADE_ORDEM = ['Júnior', 'Pleno', 'Sênior', 'Gestor']

CARGO_MAP = {
    'Analista de Dados/Data Analyst':                    'Analista de Dados',
    'Analista de BI/BI Analyst':                         'Analista de Dados',
    'Analista de BI/BI Analyst/Analytics Engineer':      'Analista de Dados',
    'Analista de Negócios/Business Analyst':             'Analista de Dados',
    'Analista de Inteligência de Mercado/Market Intelligence': 'Analista de Dados',
    'Analista de Marketing':                             'Analista de Dados',
    'Analista Administrativo':                           'Analista de Dados',
    'Estatístico':                                       'Analista de Dados',
    'Economista':                                        'Analista de Dados',
    'Engenheiro de Dados/Arquiteto de Dados/Data Engineer/Data Architect': 'Engenheiro de Dados',
    'Engenheiro de Dados/Data Engineer':                 'Engenheiro de Dados',
    'Analytics Engineer':                                'Engenheiro de Dados',
    'Arquiteto de Dados':                                'Engenheiro de Dados',
    'Arquiteto de dados':                                'Engenheiro de Dados',
    'DBA/Administrador de Banco de Dados':               'Engenheiro de Dados',
    'Engenheiro de Dados/Data Engineer/Data Architect':  'Engenheiro de Dados',
    'Arquiteto de Dados/Data Architect':                 'Engenheiro de Dados',
    'Cientista de Dados/Data Scientist':                 'Cientista de Dados',
    'Engenheiro de Machine Learning/ML Engineer':        'Cientista de Dados',
    'Engenheiro de Machine Learning/ML Engineer/AI Engineer': 'Cientista de Dados',
    'Professor':                                         'Professor/Pesquisador',
    'Data Product Manager/ Product Manager (PM/APM/DPM/GPM/PO)': 'Product Manager',
    'Product Manager/ Product Owner (PM/APM/DPM/GPM/PO)': 'Product Manager',
    'Desenvolvedor/ Engenheiro de Software/ Analista de Sistemas': 'Desenvolvedor',
    'Desenvolvedor ou Engenheiro de Software':           'Desenvolvedor',
    'Analista de Sistemas/Analista de TI':               'Desenvolvedor',
    'Analista de Suporte/Analista Técnico':              'Desenvolvedor',
    'Suporte Técnico':                                   'Desenvolvedor',
    'Técnico':                                           'Desenvolvedor',
    'Outra Opção':                                       'Outro',
    'Outras Engenharias (não inclui dev)':               'Outro',
}
AREA_FORMACAO_MAP = {
    'Computação / Engenharia de Software / Sistemas de Informação/ TI': 'Computação / TI',
    'Outras Engenharias': 'Engenharia (outras)',
    'Outras Engenharias (não incluir engenharia de software ou TI)': 'Engenharia (outras)',
    'Economia/ Administração / Contabilidade / Finanças/ Negócios': 'Economia / Adm / Finanças',
    'Economia/ Administração / Contabilidade / Finanças': 'Economia / Adm / Finanças',
    'Estatística/ Matemática / Matemática Computacional/ Ciências Atuariais': 'Estatística / Matemática',
    'Estatística/ Matemática / Matemática Computacional': 'Estatística / Matemática',
    'Ciências Biológicas/ Farmácia/ Medicina/ Área da Saúde': 'Ciências da Saúde',
    'Ciências Biológicas/Farmácia/Medicina/Área da Saúde': 'Ciências da Saúde',
    'Marketing / Publicidade / Comunicação / Jornalismo': 'Marketing / Comunicação',
    'Marketing / Publicidade / Comunicação / Jornalismo / Ciências Sociais': 'Marketing / Comunicação',
    'Outra opção': 'Outras',
}
SITUACAO_MAP = {
    'Vivo no Brasil e trabalho remoto para empresa de fora do Brasil':      'Remoto p/ exterior',
    'Vivo no Brasil e trabalho remoto para empresa de fora do Brasil (PJ)': 'Remoto p/ exterior',
    'Desempregado, buscando recolocação':                                   'Desempregado',
    'Desempregado e não estou buscando recolocação':                        'Desempregado',
    'Somente Estudante (graduação)':                                        'Somente Estudante',
    'Somente Estudante (pós-graduação)':                                    'Somente Estudante',
    'Prefiro não informar':                                                  None,
}

REGIAO_MAP = {
    'Centro_Oeste': 'Centro-Oeste', 'Centro Oeste': 'Centro-Oeste',
    'centro-oeste': 'Centro-Oeste', 'Centro-oeste': 'Centro-Oeste',
    'nordeste': 'Nordeste', 'norte': 'Norte',
    'sudeste': 'Sudeste',  'sul':    'Sul',
    'Exterior': None, 'Prefiro não informar': None,
}

ANOS     = ['2021', '2022', '2023', '2024']
REGIOES = ['Sudeste', 'Sul', 'Nordeste', 'Centro-Oeste', 'Norte']

CORES_GENERO = {'Masculino': '#517493', 'Feminino': '#64313E'}
CORES_REGIAO = {
    'Sudeste':      '#2563EB',
    'Sul':          '#16A34A',
    'Nordeste':     '#EA580C',
    'Centro-Oeste': '#9333EA',
    'Norte':        '#CA8A04',
}
TEMPLATE = 'plotly_white'

---
## Seção 1 — Carregamento e Pipeline de Pré-processamento

In [30]:
def build_year_dataframe(df_raw: pd.DataFrame, year: str) -> pd.DataFrame:

    # COLUNAS BASE
    
    base_map = COLUMN_MAPPING[year]

    cols_base = {
        old: new
        for new, old in base_map.items()
        if old in df_raw.columns
    }

    df = df_raw[list(cols_base.keys())].rename(columns=cols_base).copy()

    
    # COLUNAS DE IA (2023/2024)
    
    if year in IA_COLS:

        ia_map = IA_COLS[year]

        cols_ia = {
            old: new
            for new, old in ia_map.items()
            if old in df_raw.columns
        }
        
        ia_df = (df_raw[list(cols_ia.keys())].rename(columns=cols_ia))

        df = pd.concat([df, ia_df], axis=1)

    # ANO
   
    df.insert(0, 'ano_pesquisa', year)

    return df
    

In [31]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:

    
    # GÊNERO
    
    df = df[df['genero'].isin(['Masculino', 'Feminino'])].copy()

    # NÍVEL DE ENSINO
    
    df['nivel_ensino'] = (
        df['nivel_ensino']
        .replace({
            'Pós-graduação': 'Especialização Lato Sensu',
            'Prefiro não informar': np.nan
        })
    )

    df['nivel_ensino'] = pd.Categorical(
        df['nivel_ensino'],
        categories=NIVEL_ENSINO_ORDEM,
        ordered=True
    )

    # FAIXA SALARIAL
    
    df['faixa_salarial'] = (
        df['faixa_salarial']
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
        .replace(FAIXA_SALARIAL_MAP)
    )

    df['faixa_salarial'] = pd.Categorical(
        df['faixa_salarial'],
        categories=FAIXA_SALARIAL_ORDEM,
        ordered=True
    )

    # CARGO / ÁREA / REGIÃO
    
    df['cargo'] = df['cargo'].replace(CARGO_MAP)

    df['area_formacao'] = (
        df['area_formacao']
        .replace(AREA_FORMACAO_MAP)
    )

    df['regiao'] = (
        df['regiao']
        .replace(REGIAO_MAP)
    )

    # SENIORIDADE
    
    df['nivel'] = pd.Categorical(
        df['nivel'].where(
            df['nivel'].isin(NIVEL_SENIORIDADE_ORDEM)
        ),
        categories=NIVEL_SENIORIDADE_ORDEM,
        ordered=True
    )

    # GESTOR (0/1)
    
    df['gestor'] = (
        df['gestor']
        .replace({
            'Sim': 1,
            'Não': 0,
            'sim': 1,
            'não': 0
        })
        .pipe(pd.to_numeric, errors='coerce')
        .fillna(0)
        .astype(int)
    )

    # SITUAÇÃO
    
    df['situacao'] = (
        df['situacao']
        .replace(SITUACAO_MAP)
    )

    df = df[df['situacao'].notna()]

    return df.reset_index(drop=True)

In [32]:
def run_pipeline(dfs_raw: dict) -> pd.DataFrame:
    """
    Constrói o dataset unificado a partir dos CSVs brutos.
    Etapas: renomear colunas → concatenar anos → limpar.
    """
    frames = []
    for year in ANOS:
        print(f'  {year}...')
        frames.append(build_year_dataframe(dfs_raw[year], year))

    df = pd.concat(frames, ignore_index=True)
    df = preprocess(df)
    return df


In [33]:
df = run_pipeline(dfs_raw)

  2021...
  2022...
  2023...
  2024...


In [34]:
# SPLIT TREINO / TESTE

df_train = df[df['ano_pesquisa'] != '2024'].copy()

df_test = df[df['ano_pesquisa'] == '2024'].copy()

# SAVE

df.to_parquet(
    'data/df_full.parquet',
    index=False
)

df_train.to_parquet(
    'data/df_train.parquet',
    index=False
)

df_test.to_parquet(
    'data/df_test.parquet',
    index=False
)

# SHAPES

print(f' df_full: {df.shape}')
print(f' treino: {df_train.shape}')
print(f' teste: {df_test.shape}')

# -----------------------------
# COBERTURA IA
# -----------------------------
print('\nCobertura IA:')

for nome, subset in [
    ('Treino', df_train),
    ('Teste', df_test)
]:

    cobertura = (
        subset[IA_FEATS_ALL]
        .notna()
        .any(axis=1)
        .mean() * 100
    )

    print(f'{nome}: {cobertura:.1f}%')

 df_full: (17295, 24)
 treino: (12117, 24)
 teste: (5178, 24)

Cobertura IA:
Treino: 38.3%
Teste: 89.5%


---
## Seção 2 — Análise Exploratória: Perfil por Gênero (2021–2024)

In [35]:
def make_prop_table(
    df,
    group_col,
    target='genero',
    normalize='target'
):
    """
    normalize:
        - 'target'  -> % dentro do gênero
        - 'group'   -> % dentro do grupo
    """

    tab = pd.crosstab(
        df[group_col],
        df[target]
    )

    axis = 1 if normalize == 'target' else 0

    return (
        tab.div(tab.sum(axis=axis), axis=1-axis) * 100
    )


def female_representation(df, group_col):

    tab = pd.crosstab(
        df[group_col],
        df['genero']
    )

    return (
        tab['Feminino'] /
        tab.sum(axis=1)
    ) * 100


def update_layout(fig, title, height=450, **kwargs):

    fig.update_layout(
        title=title,
        template=TEMPLATE,
        height=height,
        legend=dict(
            orientation='h',
            y=1.02,
            x=1,
            xanchor='right'
        ),
        **kwargs
    )

    return fig


def remove_duplicate_legend(fig):

    seen = set()

    for trace in fig.data:

        trace.showlegend = trace.name not in seen

        seen.add(trace.name)

    return fig

In [36]:
def temporal_barplot(
    df,
    category,
    title,
    categories_order=None,
    percent=False,
    pyramid=False,
    height=450
):

    fig = make_subplots(
        rows=1,
        cols=len(ANOS),
        subplot_titles=ANOS,
        shared_yaxes=True
    )

    for i, ano in enumerate(ANOS, start=1):

        sub = df[df['ano_pesquisa'] == ano]

        tab = pd.crosstab(
            sub[category],
            sub['genero']
        ).fillna(0)

        if categories_order:
            tab = tab.reindex(categories_order)

        if percent:
            tab = tab.div(tab.sum(axis=0), axis=1) * 100

        for genero in ['Feminino', 'Masculino']:

            vals = tab.get(
                genero,
                pd.Series([0]*len(tab), index=tab.index)
            )

            x = -vals if pyramid and genero == 'Feminino' else vals

            fig.add_trace(
                go.Bar(
                    y=tab.index,
                    x=x,
                    orientation='h',
                    name=genero,
                    marker_color=CORES_GENERO[genero],
                    text=np.round(vals, 1),
                    textposition='auto'
                ),
                row=1,
                col=i
            )

        if pyramid:

            fig.add_shape(
                type='line',
                x0=0,
                x1=0,
                y0=-0.5,
                y1=len(tab)-0.5,
                line=dict(color='black', dash='dot'),
                row=1,
                col=i
            )

    fig = remove_duplicate_legend(fig)

    fig = update_layout(
        fig,
        title,
        height=height,
        barmode='relative' if pyramid else 'group'
    )

    return fig

In [37]:
fig = temporal_barplot(
    df=df,
    category='faixa_salarial',
    title='Distribuição de Faixa Salarial por Gênero',
    categories_order=FAIXA_SALARIAL_ORDEM,
    height=550
)

fig.show()

In [38]:
fig = temporal_barplot(
    df=df,
    category='nivel',
    title='Pirâmide de Senioridade por Gênero',
    categories_order=NIVEL_SENIORIDADE_ORDEM,
    percent=True,
    pyramid=True,
    height=400
)

fig.show()

In [39]:
evolucao = (
    female_representation(df, 'ano_pesquisa')
    .reindex(ANOS)
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=ANOS,
        y=evolucao,
        mode='lines+markers+text',
        text=[f'{v:.1f}%' for v in evolucao],
        textposition='top center',
        line=dict(
            color=CORES_GENERO['Feminino'],
            width=3
        ),
        marker=dict(size=9),
        name='% Feminino'
    )
)

fig = update_layout(
    fig,
    'Evolução da Representatividade Feminina',
    height=350
)

fig.update_yaxes(
    ticksuffix='%',
    title='% Feminino'
)

fig.show()

In [40]:
def ia_adoption_table(df, cols):

    return (
        df.groupby('genero')[cols]
        .mean()
        .mul(100)
    )

In [41]:
def plot_ia_adoption(tab, title):

    delta = (
        tab.loc['Masculino'] -
        tab.loc['Feminino']
    ).abs()

    order = delta.sort_values().index

    tab = tab[order]

    ax = tab.T.plot.barh(figsize=(10,6))

    ax.set_title(title)

    plt.tight_layout()
    plt.show()

---
## Seção 3 — Análise Exploratória: Recorte Regional

In [42]:
df_reg = df[df['regiao'].isin(REGIOES)].copy()
df_reg['regiao'] = pd.Categorical(df_reg['regiao'], categories=REGIOES, ordered=False)
print(f'Linhas com região válida: {len(df_reg):,}')
print(df_reg.groupby(['ano_pesquisa', 'regiao'], observed=True).size().unstack(fill_value=0).to_string())

Linhas com região válida: 16,884
regiao        Sudeste   Sul  Nordeste  Centro-Oeste  Norte
ano_pesquisa                                              
2021             1663   398       300           183     36
2022             2618   662       558           217     75
2023             3147   958       608           342     82
2024             3114  1026       505           329     63


In [43]:
# G-R1 — Representatividade Feminina por Região e Ano
fig = go.Figure()
for regiao in REGIOES:
    pcts = []
    for ano in ANOS:
        sub = df_reg[(df_reg['regiao'] == regiao) & (df_reg['ano_pesquisa'] == ano)]
        ct  = sub['genero'].value_counts()
        fem = ct.get('Feminino', 0); tot = fem + ct.get('Masculino', 0)
        pcts.append(round(fem/tot*100, 1) if tot > 0 else None)
    fig.add_trace(go.Scatter(
        x=ANOS, y=pcts, mode='lines+markers+text', name=regiao,
        line=dict(color=CORES_REGIAO[regiao], width=2.5), marker=dict(size=9),
        text=[f'{v}%' if v else '' for v in pcts], textposition='top center',
    ))
fig = layout_padrao(fig, '% de Respondentes Femininas por Região (2021–2024)', height=420)
fig.update_yaxes(ticksuffix='%', range=[0, 40])
fig.show()

NameError: name 'layout_padrao' is not defined

In [ ]:
# G-R2 — Heatmap % Feminino por Região × Ano
matriz = pd.DataFrame(index=REGIOES, columns=ANOS, dtype=float)
for regiao in REGIOES:
    for ano in ANOS:
        sub = df_reg[(df_reg['regiao'] == regiao) & (df_reg['ano_pesquisa'] == ano)]
        ct  = sub['genero'].value_counts()
        fem = ct.get('Feminino', 0); tot = fem + ct.get('Masculino', 0)
        matriz.loc[regiao, ano] = round(fem/tot*100, 1) if tot > 0 else np.nan
fig = go.Figure(go.Heatmap(
    z=matriz.values.astype(float), x=ANOS, y=REGIOES,
    text=[[f'{v:.1f}%' for v in row] for row in matriz.values.astype(float)],
    texttemplate='%{text}', colorscale='RdBu', zmid=20, zmin=0, zmax=40,
    colorbar=dict(title='% Feminino', ticksuffix='%'),
))
fig = layout_padrao(fig, 'Heatmap: % Feminino por Região e Ano', height=360)
fig.show()

---
## Seção 4 — Análise Estatística Inferencial

In [ ]:
df['genero_bin']         = (df['genero'] == 'Feminino').astype(int)
df['faixa_salarial_num'] = df['faixa_salarial'].cat.codes.replace(-1, np.nan)
df['ano_num']            = df['ano_pesquisa'].astype(int)
df['ano_centro']         = df['ano_num'] - 2021

variaveis = {
    'Nível de Senioridade': 'nivel',
    'Faixa Salarial':       'faixa_salarial',
    'Nível de Ensino':      'nivel_ensino',
    'Cargo':                'cargo',
    'Modalidade':           'modalidade',
    'Situação':             'situacao',
    'Área de Formação':     'area_formacao',
    'Região':               'regiao',
}
print(f'N total: {len(df):,}')
print(f'N Feminino:  {(df["genero"]=="Feminino").sum():,} ({(df["genero"]=="Feminino").mean()*100:.1f}%)')
print(f'N Masculino: {(df["genero"]=="Masculino").sum():,} ({(df["genero"]=="Masculino").mean()*100:.1f}%)')

N total: 17,295
N Feminino:  4,055 (23.4%)
N Masculino: 13,240 (76.6%)


In [ ]:
# 4.2 Qui-quadrado + V de Cramér
def cramers_v(ct):
    chi2, _, _, _ = chi2_contingency(ct)
    n = ct.sum().sum(); k = min(ct.shape) - 1
    return np.sqrt(chi2 / (n*k)) if k > 0 and n > 0 else 0.0

def teste_qui_quadrado(df, col, label):
    sub = df.dropna(subset=[col, 'genero'])
    ct  = pd.crosstab(sub[col], sub['genero'])
    chi2, p, dof, expected = chi2_contingency(ct)
    v = cramers_v(ct)
    return {
        'Variável':          label,
        'N':                 len(sub),
        'χ²':                round(chi2, 3),
        'gl':                dof,
        'p-valor':           round(p, 6),
        'Significativo':     'Sim' if p < ALPHA else 'Não',
        'V de Cramér':       round(v, 4),
        'Magnitude':         ('Forte' if v >= .4 else 'Moderado' if v >= .2 else 'Pequeno' if v >= .1 else 'Negligenciável'),
        '% células (exp≥5)': f'{(expected >= 5).mean()*100:.0f}%',
    }

tab_chi2 = pd.DataFrame([teste_qui_quadrado(df, col, label) for label, col in variaveis.items()]).set_index('Variável')
print('=== Qui-quadrado + V de Cramér ===')
print(tab_chi2.to_string())

=== Qui-quadrado + V de Cramér ===
                          N       χ²  gl   p-valor Significativo  V de Cramér       Magnitude % células (exp≥5)
Variável                                                                                                       
Nível de Senioridade  15557  104.490   3  0.000000           Sim       0.0820  Negligenciável              100%
Faixa Salarial        15438  184.015  11  0.000000           Sim       0.1092         Pequeno              100%
Nível de Ensino       17269   89.471   5  0.000000           Sim       0.0720  Negligenciável              100%
Cargo                 12422  139.500   6  0.000000           Sim       0.1060         Pequeno              100%
Modalidade            15557   60.570   3  0.000000           Sim       0.0624  Negligenciável              100%
Situação              17295  118.078   9  0.000000           Sim       0.0826  Negligenciável              100%
Área de Formação      16910  539.484   8  0.000000           Sim     

In [ ]:
# 4.3 Mann-Whitney U — Desigualdade Salarial
def mann_whitney_salarial(df_sub, label='Geral'):
    sub  = df_sub.dropna(subset=['faixa_salarial_num', 'genero'])
    masc = sub[sub['genero'] == 'Masculino']['faixa_salarial_num']
    fem  = sub[sub['genero'] == 'Feminino']['faixa_salarial_num']
    if len(masc) == 0 or len(fem) == 0: return None
    U, p = mannwhitneyu(masc, fem, alternative='greater')
    n = len(masc) + len(fem)
    r = abs(stats.norm.ppf(1 - p)) / np.sqrt(n)
    return {
        'Grupo': label, 'N Masculino': len(masc), 'N Feminino': len(fem),
        'Mediana Masc (rank)': round(masc.median(), 1),
        'Mediana Fem (rank)':  round(fem.median(),  1),
        'Faixa modal Masc': FAIXA_SALARIAL_ORDEM[int(masc.mode()[0])],
        'Faixa modal Fem':  FAIXA_SALARIAL_ORDEM[int(fem.mode()[0])],
        'U': round(U), 'p-valor': round(p, 6),
        'Significativo':  'Sim' if p < ALPHA else 'Não',
        'r (effect size)': round(r, 4),
        'Magnitude r': ('Grande' if r>=.5 else 'Médio' if r>=.3 else 'Pequeno' if r>=.1 else 'Negligenciável'),
    }

res_mw = [mann_whitney_salarial(df, 'Brasil — Geral')]
for ano in ANOS:
    res_mw.append(mann_whitney_salarial(df[df['ano_pesquisa'] == ano], f'Brasil — {ano}'))
for reg in sorted(df['regiao'].dropna().unique()):
    res_mw.append(mann_whitney_salarial(df[df['regiao'] == reg], f'Região: {reg}'))

tab_mw = pd.DataFrame([r for r in res_mw if r]).set_index('Grupo')
print('=== Mann-Whitney U: Diferença Salarial por Gênero ===')
print(tab_mw.to_string())

=== Mann-Whitney U: Diferença Salarial por Gênero ===
                      N Masculino  N Feminino  Mediana Masc (rank)  Mediana Fem (rank) Faixa modal Masc Faixa modal Fem         U   p-valor Significativo  r (effect size)     Magnitude r
Grupo                                                                                                                                                                                     
Brasil — Geral              11900        3538                  5.0                 4.0         R$8k-12k        R$8k-12k  23712840  0.000000           Sim              inf          Grande
Brasil — 2021                1892         430                  4.0                 4.0         R$8k-12k        R$8k-12k    454266  0.000065           Sim           0.0794  Negligenciável
Brasil — 2022                2749         876                  5.0                 4.0         R$8k-12k        R$8k-12k   1349824  0.000000           Sim           0.0906  Negligenciável
Brasil — 20

In [ ]:
# 4.4 Regressão Logística Simples — Tendência Temporal
def regressao_temporal(df_sub, label='Brasil'):
    sub = df_sub.dropna(subset=['genero_bin', 'ano_centro'])
    if len(sub) < 50: return None
    X   = sm.add_constant(sub['ano_centro'])
    mod = sm.Logit(sub['genero_bin'], X).fit(disp=False)
    coef = mod.params['ano_centro']; p = mod.pvalues['ano_centro']
    OR = np.exp(coef); ci = mod.conf_int().loc['ano_centro']
    return {
        'Grupo': label, 'β (log-odds/ano)': round(coef, 4), 'OR': round(OR, 4),
        'IC 95% OR': f'[{np.exp(ci[0]):.3f}, {np.exp(ci[1]):.3f}]',
        'p-valor': round(p, 6), 'Significativo': 'Sim' if p < ALPHA else 'Não',
        'Tendência': ('Crescente' if coef > 0 and p < ALPHA else 'Decrescente' if coef < 0 and p < ALPHA else 'Estável'),
    }

res_temp = [regressao_temporal(df, 'Brasil — Geral')]
for reg in sorted(df['regiao'].dropna().unique()):
    res_temp.append(regressao_temporal(df[df['regiao'] == reg], f'Região: {reg}'))
tab_temp = pd.DataFrame([r for r in res_temp if r]).set_index('Grupo')
print('=== Tendência Temporal da Representatividade Feminina ===')
print(tab_temp.to_string())

=== Tendência Temporal da Representatividade Feminina ===
                      β (log-odds/ano)      OR       IC 95% OR   p-valor Significativo  Tendência
Grupo                                                                                            
Brasil — Geral                  0.0585  1.0603  [1.025, 1.097]  0.000711           Sim  Crescente
Região: Centro-Oeste            0.1135  1.1202  [0.971, 1.292]  0.119501           Não    Estável
Região: Nordeste                0.0407  1.0415  [0.940, 1.155]  0.439353           Não    Estável
Região: Norte                  -0.0507  0.9505  [0.691, 1.308]  0.755645           Não    Estável
Região: Sudeste                 0.0557  1.0573  [1.013, 1.103]  0.010410           Sim  Crescente
Região: Sul                     0.0722  1.0748  [0.989, 1.169]  0.090758           Não    Estável


In [ ]:
# Exporta tabelas estatísticas
with pd.ExcelWriter('resultados_estatisticos.xlsx') as writer:
    tab_chi2.to_excel(writer, sheet_name='Qui-quadrado e Cramer V')
    tab_mw.to_excel(writer,   sheet_name='Mann-Whitney Salarial')
    tab_temp.to_excel(writer, sheet_name='Tendência Temporal')
print('✅ resultados_estatisticos.xlsx salvo')

✅ resultados_estatisticos.xlsx salvo


---
## Seção 5 — Modelagem Preditiva (Machine Learning)

### Estratégia

| Aspecto | Decisão |
|---|---|
| Treino | 2021–2023 (**inclui features de IA de 2023**) |
| Teste  | 2024 (**inclui features de IA de 2024**) |
| Targets | Faixa Salarial · Senioridade · Gestor/a |
| Modelos | Logistic Regression · Random Forest · XGBoost |
| Seleção | Melhor F1-macro em CV 5-fold estratificado |

> **Modelo A:** treino 2021–2023 sem IA / teste 2024 sem IA — baseline  
> **Modelo B:** treino 2021–2023 **com IA de 2023** / teste 2024 **com IA de 2024** — comparação honesta
>
> A comparação A vs B responde: **"IA generativa é preditor relevante de salário/senioridade?"**

In [ ]:
BASE_FEATURES = [
    'nivel_ensino', 'area_formacao', 'cargo',
    'tempo_area_dados', 'tempo_area_ti', 'modalidade',
    'setor', 'regiao', 'situacao',
]
TARGETS = {
    'faixa_salarial': 'multiclass',
    'nivel':          'multiclass',
    'gestor':         'binario',
}
TARGET_LABELS = {
    'faixa_salarial': 'Faixa Salarial',
    'nivel':          'Senioridade',
    'gestor':         'Gestor/a',
}

def encode_ordinal(series, ordem):
    cat   = pd.Categorical(series, categories=ordem, ordered=True)
    codes = cat.codes.astype(float)
    codes[codes == -1] = np.nan
    return codes

def prepare_xy(df: pd.DataFrame, target: str, extra_features: list = None):
    feats = BASE_FEATURES.copy()
    if extra_features:
        feats += [f for f in extra_features if f in df.columns]
    sub = df[feats + [target, 'ano_pesquisa', 'genero']].copy()
    sub['nivel_ensino'] = encode_ordinal(sub['nivel_ensino'], NIVEL_ENSINO_ORDEM)
    if target == 'faixa_salarial':
        sub['y'] = encode_ordinal(sub['faixa_salarial'], FAIXA_SALARIAL_ORDEM)
    elif target == 'nivel':
        sub['y'] = encode_ordinal(sub['nivel'], NIVEL_SENIORIDADE_ORDEM)
    elif target == 'gestor':
        sub['y'] = (sub['gestor'] == 1.0).astype(float)
    sub = sub.dropna(subset=['y'])
    sub['y'] = sub['y'].astype(int)
    return sub[feats].copy(), sub['y'], sub[['genero', 'ano_pesquisa']].copy()

def build_preprocessor(X):
    num_cols = X.select_dtypes(include='number').columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    transformers = []
    if num_cols:
        transformers.append(('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')), ('scl', StandardScaler()),
        ]), num_cols))
    if cat_cols:
        transformers.append(('cat', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), cat_cols))
    return ColumnTransformer(transformers, remainder='drop')

def get_models(task_type):
    if task_type == 'binario':
        return {
            'Regressão Logística': LogisticRegression(max_iter=500, random_state=SEED, class_weight='balanced'),
            'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=SEED, class_weight='balanced', n_jobs=-1),
            'XGBoost':             XGBClassifier(n_estimators=200, random_state=SEED, eval_metric='logloss', scale_pos_weight=5, verbosity=0),
        }
    return {
        'Regressão Logística': LogisticRegression(max_iter=500, random_state=SEED, class_weight='balanced'),
        'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=SEED, class_weight='balanced', n_jobs=-1),
        'XGBoost':             XGBClassifier(n_estimators=200, random_state=SEED, eval_metric='mlogloss', verbosity=0),
    }

def run_cv(X, y, models, cv=5):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=SEED)
    results = []
    for name, model in models.items():
        pipe   = Pipeline([('pre', build_preprocessor(X)), ('clf', model)])
        scores = cross_val_score(pipe, X, y, cv=skf, scoring='f1_macro', n_jobs=-1)
        results.append({'Modelo': name, 'F1_macro_mean': scores.mean(), 'F1_macro_std': scores.std()})
        print(f'    {name:25s}: {scores.mean():.3f} ± {scores.std():.3f}')
    return pd.DataFrame(results)

# Quando o target é gestor, remove 'nivel' das features.
# Motivo: nivel='Gestor' e gestor=1 são altamente correlacionados —
# manter nivel tornaria o target trivialmente predizível.
if target == 'gestor' and 'nivel' in feats:
    feats = [f for f in feats if f != 'nivel']
    
def treinar_avaliar(target, task_type, df_train, df_test, extra_features=None, label_extra=''):
    tag = f'{TARGET_LABELS[target]}{" " + label_extra if label_extra else ""}'
    print(f'\n{"="*60}\n  {tag}\n{"="*60}')
    X_train, y_train, _         = prepare_xy(df_train, target, extra_features)
    X_test,  y_test,  meta_test = prepare_xy(df_test,  target, extra_features)
    print(f'  Treino: {len(X_train):,} | Teste: {len(X_test):,} | Classes: {sorted(y_train.unique())}')
    print(f'\n  [CV 5-fold — F1-macro]')
    models    = get_models(task_type)
    cv_df     = run_cv(X_train, y_train, models)
    best_name = cv_df.loc[cv_df['F1_macro_mean'].idxmax(), 'Modelo']
    print(f'\n  Melhor modelo: {best_name}')
    pipe = Pipeline([('pre', build_preprocessor(X_train)), ('clf', models[best_name])])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    print(f'\n  [Classificação — Teste]')
    print(classification_report(y_test, y_pred, zero_division=0))
    gender_rows = []
    for g in ['Masculino', 'Feminino']:
        mask = meta_test['genero'] == g
        if mask.sum() == 0: continue
        gender_rows.append({
            'Gênero': g, 'N': int(mask.sum()),
            'Acurácia': round(accuracy_score(y_test[mask], y_pred[mask]), 4),
            'F1_macro': round(f1_score(y_test[mask], y_pred[mask], average='macro', zero_division=0), 4),
        })
    print(f'\n  [Fairness por Gênero]')
    print(pd.DataFrame(gender_rows).to_string(index=False))
    return {
        'target': target, 'label': tag, 'best_name': best_name, 'pipe': pipe,
        'cv_df': cv_df, 'X_train': X_train, 'X_test': X_test,
        'y_test': y_test, 'y_pred': y_pred, 'gender_rows': gender_rows, 'meta_test': meta_test,
    }



✅ Pipeline de modelos definido


In [ ]:
df_train = pd.read_parquet('data/df_train.parquet')
df_test  = pd.read_parquet('data/df_test.parquet')

print(f'df_train: {len(df_train):,} linhas — anos {sorted(df_train["ano_pesquisa"].unique())}')
print(f'df_test:  {len(df_test):,}  linhas — anos {sorted(df_test["ano_pesquisa"].unique())}')

# Features de IA disponíveis e preenchidas em cada split
ia_treino = [c for c in IA_FEATS_ALL if c in df_train.columns and df_train[c].notna().sum() > 0]
ia_teste  = [c for c in IA_FEATS_ALL if c in df_test.columns  and df_test[c].notna().sum()  > 0]
print(f'\nIA no treino: {len(ia_treino)}/{len(IA_FEATS_ALL)} features ({df_train[ia_treino].notna().any(axis=1).mean()*100:.1f}% das linhas preenchidas)')
print(f'IA no teste:  {len(ia_teste)}/{len(IA_FEATS_ALL)} features ({df_test[ia_teste].notna().any(axis=1).mean()*100:.1f}% das linhas preenchidas)')

df_train: 12,117 linhas — anos ['2021', '2022', '2023']
df_test:  5,178  linhas — anos ['2024']

IA no treino: 0/5 features (0.0% das linhas preenchidas)
IA no teste:  0/5 features (0.0% das linhas preenchidas)


In [ ]:
# Modelo A — Treino 2021–2023 / Teste 2024 — SEM features de IA (baseline)
results_A = {}
for target, task_type in TARGETS.items():
    results_A[target] = treinar_avaliar(
        target, task_type, df_train, df_test,
        extra_features=None, label_extra='[A — sem IA]',
    )


  Faixa Salarial [A — sem IA]
  Treino: 10,643 | Teste: 4,795 | Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)]

  [CV 5-fold — F1-macro]
    Regressão Logística      : 0.262 ± 0.007
    Random Forest            : 0.265 ± 0.011
    XGBoost                  : 0.262 ± 0.010

  Melhor modelo: Random Forest

  [Classificação — Teste]
              precision    recall  f1-score   support

           0       0.51      0.68      0.58       151
           1       0.26      0.23      0.25       235
           2       0.19      0.16      0.18       268
           3       0.31      0.35      0.33       592
           4       0.23      0.18      0.20       649
           5       0.37      0.50      0.42      1070
           6       0.30      0.34      0.32       714
           7       0.27      0.16      0.20       453
           8       0.18      0.13      0.15       246
      

In [ ]:
# Modelo B — Treino 2021–2023 (IA de 2023) / Teste 2024 (IA de 2024)
# Usa apenas as features de IA que têm dados tanto no treino quanto no teste
ia_comum = [c for c in IA_FEATS_ALL if c in ia_treino and c in ia_teste]
print(f'Features de IA comuns (treino ∩ teste): {len(ia_comum)}')
print()

results_B = {}
for target, task_type in TARGETS.items():
    results_B[target] = treinar_avaliar(
        target, task_type, df_train, df_test,
        extra_features=ia_comum, label_extra='[B — base + IA]',
    )

Features de IA comuns (treino ∩ teste): 0


  Faixa Salarial [B — base + IA]
  Treino: 10,643 | Teste: 4,795 | Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)]

  [CV 5-fold — F1-macro]
    Regressão Logística      : 0.262 ± 0.007
    Random Forest            : 0.265 ± 0.011
    XGBoost                  : 0.262 ± 0.010

  Melhor modelo: Random Forest

  [Classificação — Teste]
              precision    recall  f1-score   support

           0       0.51      0.68      0.58       151
           1       0.26      0.23      0.25       235
           2       0.19      0.16      0.18       268
           3       0.31      0.35      0.33       592
           4       0.23      0.18      0.20       649
           5       0.37      0.50      0.42      1070
           6       0.30      0.34      0.32       714
           7       0.27      0.16      0.20       453
           8  

---
## Seção 6 — Explicabilidade com SHAP

In [ ]:
def compute_shap(pipe, X, model_name, max_samples=600):
    pre = pipe.named_steps['pre']; clf = pipe.named_steps['clf']
    X_t = pre.transform(X)
    try:
        feat_names = np.array(pre.get_feature_names_out())
    except Exception:
        feat_names = np.array([f'f{i}' for i in range(X_t.shape[1])])
    n   = min(max_samples, X_t.shape[0])
    idx = np.random.RandomState(SEED).choice(X_t.shape[0], n, replace=False)
    X_s = X_t[idx]
    if 'XGBoost' in model_name or 'Random Forest' in model_name:
        sv = shap.TreeExplainer(clf).shap_values(X_s)
    else:
        sv = shap.LinearExplainer(clf, X_s, feature_dependence='independent').shap_values(X_s)
    mean_abs = np.mean([np.abs(s) for s in sv], axis=0) if isinstance(sv, list) else np.abs(sv)
    return mean_abs.mean(axis=0), feat_names

def plot_shap_bar(mean_shap, feat_names, top_n=15, title='SHAP', fname='shap.png', highlight_ia=False):
    top_idx = np.argsort(mean_shap)[::-1][:top_n]
    names   = feat_names[top_idx]; vals = mean_shap[top_idx]
    clean   = lambda n: n.replace('cat__', '').replace('num__', '')
    names_c = [clean(n) for n in names]
    colors  = ['#F4A43B' if 'ia_' in n else '#4C8FBF' for n in names_c] if highlight_ia else ['#4C8FBF']*len(names_c)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(names_c[::-1], vals[::-1], color=colors[::-1])
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title(title, fontweight='bold', loc='left')
    if highlight_ia:
        ax.legend(handles=[
            mpatches.Patch(color='#F4A43B', label='Feature de IA generativa'),
            mpatches.Patch(color='#4C8FBF', label='Feature base'),
        ], loc='lower right', frameon=False)
    plt.tight_layout()
    plt.savefig(f'figures/{fname}', bbox_inches='tight', dpi=150)
    plt.show()
    print(f'  💾 figures/{fname}')

print('✅ Funções SHAP definidas')

✅ Funções SHAP definidas


In [ ]:
shap_cache = {}
for target in TARGETS:
    safe = target.replace(' ', '_')
    for key, results, hl in [('A', results_A, False), ('B', results_B, True)]:
        try:
            r = results[target]
            ms, fn = compute_shap(r['pipe'], r['X_test'], r['best_name'])
            shap_cache[f'{key}_{target}'] = (ms, fn)
            plot_shap_bar(ms, fn,
                title=f'SHAP — {TARGET_LABELS[target]} · Modelo {key} ({r["best_name"]})',
                fname=f'shap_{key}_{safe}.png', highlight_ia=hl)
        except Exception as e:
            print(f'⚠️ SHAP Modelo {key} / {target}: {e}')

In [ ]:
def plot_shap_comparacao(target, top_n=12):
    """Compara lado a lado o SHAP do Modelo A (sem IA) e B (com IA)."""
    key_a, key_b = f'A_{target}', f'B_{target}'
    if key_a not in shap_cache or key_b not in shap_cache:
        print(f'⚠️ SHAP não disponível para {target}'); return
    clean = lambda n: n.replace('cat__', '').replace('num__', '')
    def top(ms, fn, n):
        idx = np.argsort(ms)[::-1][:n]
        return [clean(fn[i]) for i in idx], ms[idx]
    ms_a, fn_a = shap_cache[key_a]; ms_b, fn_b = shap_cache[key_b]
    names_a, vals_a = top(ms_a, fn_a, top_n)
    names_b, vals_b = top(ms_b, fn_b, top_n)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, names, vals, subtitle, hl in [
        (axes[0], names_a, vals_a, 'Modelo A (sem IA)', False),
        (axes[1], names_b, vals_b, 'Modelo B (com IA de 2023 e 2024)', True),
    ]:
        colors = ['#F4A43B' if 'ia_' in n else '#4C8FBF' for n in names] if hl else ['#4C8FBF']*len(names)
        ax.barh(names[::-1], vals[::-1], color=colors[::-1])
        ax.set_xlabel('Mean |SHAP value|'); ax.set_title(subtitle, fontweight='bold')
    fig.suptitle(f'SHAP Comparativo — {TARGET_LABELS[target]}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    fname = f'figures/shap_comp_{target}.png'
    plt.savefig(fname, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'  💾 {fname}')

for target in TARGETS:
    plot_shap_comparacao(target)

---
## Observações Finais

### Sobre a inclusão de IA em 2023 no Modelo B

A comparação A vs B é metodologicamente honesta: ambos os modelos usam o mesmo split temporal (treino 2021–2023, teste 2024). A única diferença é a presença das features de IA. Como 2021 e 2022 não têm essas perguntas, o modelo aprende a relação IA × target apenas com os respondentes de 2023 e generaliza para 2024. Isso evita o viés de data leakage (treinar com 2024) e torna a comparação interpretável: se o Modelo B superar o A, a IA generativa tem poder preditivo real sobre salário e senioridade.

### Limitações
- Survey por conveniência — não probabilístico.
- Gêneros não-binários excluídos para viabilizar a comparação binária.
- SHAP calculado com amostragem (max_samples=600) por eficiência.
- Imputação por moda/mediana nos nulos das features de IA (2021–2022) pode subestimar o efeito.
